# All-concentration recovery

Recovers one RNA-associated spectrum from a full concentration series, using the
known DNA-probe spectrum as the only reference. This is the benchmark the other
recovery modes are measured against.

**Input** — `pos_path`: one CSV holding the concentration-averaged hybridized
spectra (wavenumbers in the first column, one column per concentration);
`bg_path`: the DNA-probe reference spectrum; `true_path`: the directly measured
RNA spectrum, used only to score the result, never to fit it.

**Output** — `extracted_spectrum_vs_epoch.csv` (recovered spectrum at each
epoch), `g_vs_epoch.csv` and `h_vs_epoch.csv` (the RNA and DNA mixing
coefficients), the recon/extraction metric CSVs, and diagnostic figures.

**Feeds** — Fig. 2A–C, and the R² = 0.983 validation against the measured RNA
reference.

**Scale factor.** Spectra are multiplied by `SCALE_FACTOR = 400` before the network sees them and the factor is divided out again before anything is written, so every saved spectrum is on the mean-normalized scale. This is a numerical-conditioning choice only: the measured spectrum, the DNA reference and the ground-truth reference are all scaled by the same constant, so the recovered coefficients and every shape-based metric are unchanged.


In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
import torch.nn.functional as F
import torch.optim as optim
import os
import matplotlib.pyplot as plt
from scipy.spatial.distance import cosine
from sklearn.metrics import r2_score
from scipy.stats import pearsonr
from pathlib import Path

In [ ]:
"""
INPUT DIRS HERE. SEP FOR BG FILE HERE IF NEEDED(NAME: 'bg_sep')
"""
META = {
    'bg_path': '/home/zhao/Jiaheng Cui/DNA RNA hybridization/code/Yingchuan-improve-extraction/data/2_DNA_RNA_simulated_data/11102025-ProbeDNA-ref.csv',
    'true_path': '/home/zhao/Jiaheng Cui/DNA RNA hybridization/code/Yingchuan-improve-extraction/data/2_DNA_RNA_simulated_data/11102025-SARSCoV2-ref.csv',
    'pos_folder': '/home/zhao/Jiaheng Cui/DNA RNA hybridization/code/Yingchuan-improve-extraction/data/3_DNA_RNA_experimental_data',
    'bg_sep': ',', 
    'output_path': '/mnt/f/Jiaheng Cui/DNA RNA hybridization/Results/12112025-improve_extraction_method_Yingchuan/12192025-Jiaheng-test-batch-jupyter-code',
    
    # Processing options:
    # 'csv_list': None, # Option 1: Set to None to process ALL csv files in pos_folder
    'csv_list': ["11102025-SARSCoV2-195-781-comma-normalized.csv", "11102025-SARSCoV2-781-3125-comma-normalized.csv", "11102025-SARSCoV2-12500-50000-comma-normalized.csv"], # Option 2: Set to a list of specific csv filenames to process only those,e.g., ["1.csv", "3.csv", "4.csv"]
}

os.makedirs(META['output_path'], exist_ok=True)
SCALE_FACTOR = 400.0

In [ ]:
# Get list of CSV files to process
pos_folder = Path(META['pos_folder'])
if META['csv_list'] is None:
    # Process all CSV files in folder
    csv_files = sorted([f for f in pos_folder.glob('*.csv')])
    print(f"Found {len(csv_files)} CSV files to process")
else:
    # Process only specified CSV files
    csv_files = [pos_folder / fname for fname in META['csv_list']]
    csv_files = [f for f in csv_files if f.exists()]
    print(f"Processing {len(csv_files)} specified CSV files")

if len(csv_files) == 0:
    raise ValueError("No CSV files found to process!")

print("Files to process:")
for f in csv_files:
    print(f"  - {f.name}")

In [ ]:
def load_data(pos_path, bg_path, true_path, bg_sep=','):
    """
    Assuming: mixture data are organized in ascending order of concentration. 
    Or use the column names to parse concentration values, then normalized to [0,1].
    """
    # Background
    bg_spectra = pd.read_csv(bg_path, sep=bg_sep)
    bg_mean = bg_spectra.iloc[:, 1].tolist()
    bg_mean_mu = np.mean(bg_mean)
    if bg_mean_mu != 0:
        bg_mean = bg_mean / bg_mean_mu
    else:
        print("[Warning] Background mean is zero!")
    
    # True
    true_data = pd.read_csv(true_path)
    true_y = true_data.iloc[:, 1].values.astype(np.float32)
    true_mu = true_y.mean()
    if true_mu != 0:
        true_y = true_y / true_mu
    else:
        print("[Warning] True signal mean is zero!")
    
    # Positive
    base_data = pd.read_csv(pos_path)
    
    col_x = base_data.columns[0]
    signal_cols = list(base_data.columns[1:])
    
    parsed_vals = []
    convertible = True
    for c in signal_cols:
        if isinstance(c, str):
            try:
                parsed_vals.append(float(c))
            except ValueError:
                convertible = False
                break
        else:
            parsed_vals.append(float(c))
    
    if convertible:
        parsed_vals = np.array(parsed_vals, dtype=np.float32)
        new_signal_cols = parsed_vals
        vmin, vmax = parsed_vals.min(), parsed_vals.max()
        if vmax > vmin:
            new_signal_cols = (parsed_vals - vmin) / (vmax - vmin)
        else:
            print("[Warning] Signal column values are constant, using linspace.")
            new_signal_cols = np.linspace(0, 1, len(signal_cols))
    else:
        new_signal_cols = np.linspace(0, 1, len(signal_cols))
    
    base_data.columns = [col_x] + list(new_signal_cols)
    
    # Mean normalization
    signal_df = base_data.iloc[:, 1:].astype(np.float32)
    col_means = signal_df.mean(axis=0)
    if (col_means == 0).any():
        print("[Warning] Some columns have zero mean!")
    else:
        base_data.iloc[:, 1:] = signal_df / col_means
    
    base_data.rename(columns={base_data.columns[0]: "x"}, inplace=True)
    df_long = base_data.melt(id_vars=["x"], var_name="s", value_name="y")
    
    return df_long, bg_mean, true_y

In [ ]:
class FourierFeatureMapping(nn.Module):
    def __init__(self, num_frequencies=6, include_input=True):
        super().__init__()
        self.num_frequencies = num_frequencies
        self.include_input = include_input
        self.freq_bands = 2.0 ** torch.arange(0, num_frequencies).float() * np.pi
    
    def forward(self, x):
        out = [x] if self.include_input else []
        for freq in self.freq_bands.to(x.device):
            out.append(torch.sin(freq * x))
            out.append(torch.cos(freq * x))
        return torch.cat(out, dim=-1)

class ResBlock(nn.Module):
    def __init__(self, dim):
        super(ResBlock, self).__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim),
            nn.ReLU(),
            nn.Linear(dim, dim)
        )
        self.activation = nn.ReLU()
    
    def forward(self, x):
        return self.activation(x + self.block(x))

class SERSDecomposition(nn.Module):
    def __init__(self, input_x_dim=13, input_s_dim=1, hidden_dim=256, z_dim=128): 
        super(SERSDecomposition, self).__init__()
        
        # f(x) branch
        self.f_input = nn.Sequential(
            nn.Linear(input_x_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )   
        self.f_blocks = nn.Sequential(
            ResBlock(hidden_dim),
            ResBlock(hidden_dim)
        )
        self.f_output = nn.Sequential(
            nn.Linear(hidden_dim, z_dim),
            nn.ReLU(),
            nn.Linear(z_dim, 1)
        )
        
        self.c_input = nn.Sequential(
            nn.Linear(input_s_dim, hidden_dim),
            nn.ReLU()
        )
        self.c_blocks = nn.Sequential(
            ResBlock(hidden_dim),
            ResBlock(hidden_dim),
            ResBlock(hidden_dim)
        )
        self.c_output = nn.Sequential(
            nn.Linear(hidden_dim, 2),
            nn.Softplus() 
        )
    
    def forward(self, x_embed, s):
        fx = self.f_input(x_embed)
        fx = self.f_blocks(fx)
        f_x = self.f_output(fx)
        
        cs = self.c_input(s)
        cs = self.c_blocks(cs)
        a_c = self.c_output(cs)
        a_s = a_c[:, 0:1]
        b_s = a_c[:, 1:2]
        
        return a_s, b_s, f_x

def weights_init(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

In [ ]:
def flatness_penalty(f_x, threshold, region_length=5):
    f_x = f_x.view(-1)
    diffs = f_x[1:] - f_x[:-1]
    sq_diffs = diffs ** 2
    sq_diffs = sq_diffs.unsqueeze(0).unsqueeze(0)
    kernel = torch.ones(1, 1, region_length, device=f_x.device) / region_length
    avg_sq = F.conv1d(sq_diffs, kernel, padding=region_length // 2).squeeze()
    penalty = torch.clamp(threshold - avg_sq, min=0.0)
    return torch.mean(penalty)

def custom_loss(y_true, a_s, b_s, f_x, bg, scale_mode=SCALE_FACTOR, 
                lambda_penalty=1.0, lambda_flat=0.05, threshold=1e-3, apply_flatness=False):
    
    threshold = threshold * (scale_mode ** 2)
    recon = a_s * f_x + b_s * bg
    mse_loss = torch.mean((y_true - recon) ** 2)
    neg_penalty = torch.mean(torch.clamp(-f_x, min=0.0)) * scale_mode 
    
    flat_pen = 0.0
    if apply_flatness:
        flat_pen = flatness_penalty(f_x, region_length=5, threshold=threshold)
    
    total_loss = (mse_loss + 
                  lambda_penalty * neg_penalty + 
                  lambda_flat * flat_pen)
                  
    return total_loss

In [ ]:
def to_1d(x):
    return x.detach().cpu().numpy().reshape(-1)

def train_single_file(pos_path, bg_path, true_path, output_dir, bg_sep=',', 
                      scale_factor=400.0, num_epochs=1000, device='cuda'):
    """
    Train model on a single CSV file and save results to output_dir
    """
    print(f"\n{'='*60}")
    print(f"Processing: {Path(pos_path).name}")
    print(f"Output to: {output_dir}")
    print(f"{'='*60}\n")
    
    os.makedirs(output_dir, exist_ok=True)
    
    # Load data
    dt, bg_mean_spectrum, true_y = load_data(pos_path, bg_path, true_path, bg_sep)
    minx, maxx = dt['x'].min(), dt['x'].max()
    
    # Prepare Data
    x = dt[['x']].values.astype(np.float32)
    y = dt[['y']].values.astype(np.float32)
    s_raw = dt['s'].values
    s_train = torch.tensor(s_raw.astype(np.float32).reshape(-1, 1)).to(device)
    
    bg = np.tile(bg_mean_spectrum, (len(y)//len(bg_mean_spectrum), 1)).reshape(-1, 1)
    x_train = torch.tensor(x, dtype=torch.float32).to(device)
    y_train = torch.tensor(y, dtype=torch.float32).to(device)
    bg_train = torch.tensor(bg, dtype=torch.float32).to(device)
    
    # Scaling
    y_train = y_train * scale_factor
    bg_train = bg_train * scale_factor
    
    # Normalize x
    x_train = (x_train - x_train.min()) / (x_train.max() - x_train.min())
    
    # Feature Mapping
    fourier_mapping = FourierFeatureMapping(num_frequencies=6).to(device)
    x_embed_train = fourier_mapping(x_train)
    
    # Model Init
    input_x_dim = x_embed_train.shape[1]
    model = SERSDecomposition(input_x_dim=input_x_dim).to(device)
    model.apply(weights_init)
    
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1000, gamma=0.5)
    
    # Loop variables
    loss_history = []
    f_final_list, g_final_list, h_final_list = [], [], []
    r2_recon_list, cos_recon_list = [], []
    metrics_extracted = {
        "r2": [],
        "cosine": [], 
        "pearson": []
    }
    
    # Training loop
    for epoch in range(num_epochs):
        model.train()
        apply_flatness = epoch >= 500
        
        a_s, b_s, f_x = model(x_embed_train, s=s_train)
        
        loss = custom_loss(
            y_train, a_s, b_s, f_x, bg_train, 
            scale_mode=scale_factor,
            apply_flatness=apply_flatness 
        )
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5)
        optimizer.step()
        scheduler.step()
        
        loss_history.append(loss.item())
        
        # Post-processing
        s_train_np = to_1d(s_train)
        bg_train_np = to_1d(bg_train)
        f_x_np = to_1d(f_x)
        a_s_np = to_1d(a_s)
        b_s_np = to_1d(b_s)
        y_train_np = to_1d(y_train)
        x_vals = dt['x'].values.astype(np.float32)
        
        unique_s = np.unique(s_train_np)
        
        mask0 = (s_train_np == unique_s[0]).squeeze()
        f0 = f_x_np[mask0]
        bg0 = bg_train_np[mask0]
        x0 = x_vals[mask0]
        
        # Area normalization
        f_area = np.trapz(f0, x=x0, axis=0)
        bg_area = np.trapz(bg0, x=x0, axis=0)
        
        f_final = f0 * (bg_area / f_area)
        bg_final = bg0
        true_final = true_y * scale_factor
        
        f_final_list.append(f_final)
        metrics_extracted["r2"].append(r2_score(true_final, f_final))
        metrics_extracted["cosine"].append(1 - cosine(true_final, f_final))
        pearson_corr, _ = pearsonr(true_final, f_final)
        metrics_extracted["pearson"].append(pearson_corr)
        
        a_concs, b_concs = [], []
        r2_concs, cos_concs = [], []
        
        for s_val in unique_s:
            mask = (s_train_np == s_val).squeeze()
            y_sub = y_train_np[mask]
            
            a_pred_sub = a_s_np[mask][0]
            b_pred_sub = b_s_np[mask][0]
            
            a_final = a_pred_sub * (f_area / bg_area)
            b_final = b_pred_sub
            
            a_concs.append(a_final)
            b_concs.append(b_final)
            
            recon_np = a_final * f_final + b_final * bg_final
            r2_concs.append(r2_score(y_sub, recon_np))
            cos_concs.append(1 - cosine(y_sub, recon_np))
        
        g_final_list.append(np.array(a_concs))
        h_final_list.append(np.array(b_concs))
        r2_recon_list.append(np.array(r2_concs))
        cos_recon_list.append(np.array(cos_concs))
        
        # Progress update
        if (epoch + 1) % 100 == 0:
            print(f"Epoch {epoch+1}/{num_epochs} | Loss: {loss.item():.6f} | "
                  f"R2: {metrics_extracted['r2'][-1]:.4f} | "
                  f"Cosine: {metrics_extracted['cosine'][-1]:.4f}")
    
    print(f"\nTraining completed! Saving outputs...")
    
    # Return all results for saving
    return {
        'x0': x0,
        'f_final_list': f_final_list,
        'g_final_list': g_final_list,
        'h_final_list': h_final_list,
        'r2_recon_list': r2_recon_list,
        'cos_recon_list': cos_recon_list,
        'metrics_extracted': metrics_extracted,
        'unique_s': unique_s,
        's_train_np': s_train_np,
        'y_train_np': y_train_np,
        'bg_final': bg_final,
        'true_final': true_final,
        'loss_history': loss_history
    }

In [ ]:
def save_outputs(results, output_dir):
    """
    Save all outputs (CSV and figures) to output_dir
    """
    x0 = results['x0']
    f_final_list = results['f_final_list']
    g_final_list = results['g_final_list']
    h_final_list = results['h_final_list']
    r2_recon_list = results['r2_recon_list']
    cos_recon_list = results['cos_recon_list']
    metrics_extracted = results['metrics_extracted']
    unique_s = results['unique_s']
    s_train_np = results['s_train_np']
    y_train_np = results['y_train_np']
    bg_final = results['bg_final']
    true_final = results['true_final']
    
    num_epochs = len(f_final_list)
    num_mixtures = len(unique_s)
    epochs = np.arange(1, num_epochs + 1)
    mixture_names = [f"mixture_{i}" for i in range(num_mixtures)]
    
    # 1. Extracted spectrum vs epoch
    df_f = pd.DataFrame(
        np.column_stack([x0] + [f / SCALE_FACTOR for f in f_final_list]),  # undo the x400 conditioning factor: saved spectra are mean-normalized
        columns=["Wavenumbers"] + [f"Epoch_{i}" for i in epochs]
    )
    df_f.to_csv(os.path.join(output_dir, "extracted_spectrum_vs_epoch.csv"), index=False)
    
    # 2. a (concentration) vs epoch
    df_a = pd.DataFrame(
        np.vstack(g_final_list),
        columns=mixture_names
    )
    df_a.insert(0, "Epoch", epochs)
    df_a.to_csv(os.path.join(output_dir, "g_vs_epoch.csv"), index=False)
    
    # 3. b (background coef) vs epoch
    df_b = pd.DataFrame(
        np.vstack(h_final_list),
        columns=mixture_names
    )
    df_b.insert(0, "Epoch", epochs)
    df_b.to_csv(os.path.join(output_dir, "h_vs_epoch.csv"), index=False)
    
    # 4. R2: reconstructed vs experimental
    df_r2_recon = pd.DataFrame(
        np.vstack(r2_recon_list),
        columns=mixture_names
    )
    df_r2_recon.insert(0, "Epoch", epochs)
    df_r2_recon.to_csv(os.path.join(output_dir, "r2_recon_vs_epoch.csv"), index=False)
    
    # 5. Cosine similarity: reconstructed vs experimental
    df_cos_recon = pd.DataFrame(
        np.vstack(cos_recon_list),
        columns=mixture_names
    )
    df_cos_recon.insert(0, "Epoch", epochs)
    df_cos_recon.to_csv(os.path.join(output_dir, "cosine_recon_vs_epoch.csv"), index=False)
    
    # 6. Extracted vs true metrics
    df_extract_metrics = pd.DataFrame(
        [
            metrics_extracted["r2"],
            metrics_extracted["cosine"],
            metrics_extracted["pearson"],
        ],
        index=["R2", "Cosine", "Pearson"],
        columns=[f"Epoch_{i}" for i in epochs]
    )
    df_extract_metrics.to_csv(
        os.path.join(output_dir, "extracted_vs_true_metrics.csv")
    )
    
    # 7. Figure i: Experimental vs reconstructed (each mixture)
    final_epoch_idx = -1
    
    for i, s_val in enumerate(unique_s):
        mask = (s_train_np == s_val)
        y_exp = y_train_np[mask]
        
        recon = (
            g_final_list[final_epoch_idx][i] * f_final_list[final_epoch_idx]
            + h_final_list[final_epoch_idx][i] * bg_final
        )
        
        plt.figure(figsize=(6, 4))
        plt.plot(x0, y_exp, label="Experimental", lw=1)
        plt.plot(x0, recon, label="Reconstructed", lw=1)
        
        plt.title(
            f"{mixture_names[i]} | "
            f"R2={r2_recon_list[final_epoch_idx][i]:.3f}, "
            f"Cos={cos_recon_list[final_epoch_idx][i]:.3f}"
        )
        plt.legend()
        plt.tight_layout()
        
        fname = f"fig_mixture_{i}_reconstruction.png"
        plt.savefig(os.path.join(output_dir, fname), dpi=300)
        plt.close()
    
    # 8. Figure ii: Extracted vs true spectrum
    plt.figure(figsize=(6, 4))
    plt.plot(x0, true_final, label="True", lw=2)
    plt.plot(x0, f_final_list[final_epoch_idx], label="Extracted", lw=2)
    
    plt.title(
        f"Extracted vs True | "
        f"R2={metrics_extracted['r2'][final_epoch_idx]:.3f}, "
        f"Cos={metrics_extracted['cosine'][final_epoch_idx]:.3f}, "
        f"Pearson={metrics_extracted['pearson'][final_epoch_idx]:.3f}"
    )
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "fig_extracted_vs_true.png"), dpi=300)
    plt.close()
    
    print(f"All outputs saved to: {output_dir}")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Get parameters from META
bg_path = META['bg_path']
true_path = META['true_path']
bg_sep = ',' if META['bg_sep'] == '' else META['bg_sep']
output_base = META['output_path']

# Process each CSV file
for csv_file in csv_files:
    # Create output subdirectory with csv filename (without .csv)
    csv_name = csv_file.stem  # Gets filename without extension
    output_dir = os.path.join(output_base, csv_name)
    
    try:
        # Train on this CSV
        results = train_single_file(
            pos_path=str(csv_file),
            bg_path=bg_path,
            true_path=true_path,
            output_dir=output_dir,
            bg_sep=bg_sep,
            scale_factor=SCALE_FACTOR,
            num_epochs=1000,
            device=device
        )
        
        # Save outputs
        save_outputs(results, output_dir)
        
        print(f"✓ Successfully completed: {csv_file.name}\n")
        
    except Exception as e:
        print(f"✗ Error processing {csv_file.name}: {str(e)}\n")
        continue

print("\n" + "="*60)
print("BATCH PROCESSING COMPLETED")
print("="*60)